# Interview Scoring System in Colab

Use the same JSON configuration and two-stage `app.py` as the local notebook. Configure `primary`, optional `secondary`, review thresholds and weights. Providers: `local`, `gemini`, `mistral_free`, `groq_free`, and `openrouter`. Keys stay in environment variables, role-specific `.key` files, or Colab Secrets, not notebook source.

Run the setup cells, edit the JSON in the repository directory, then run scoring. No model call is made during setup. The shipped configuration selects local inference; choose a cloud provider explicitly if wanted.

## Obtain the repository

While the two-stage scoring PR is open, the default ref below is its feature branch. After it is merged, set `REPO_REF = 'main'`. An existing checkout is reused without resetting files or changing branches, so your edits and data are preserved.

If using a Colab local runtime, set `REPO_DIR` to your existing local checkout instead of `/content/...`. Do not put credentials in clone URLs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/buriansandor/interview_scoring_system.git'
REPO_REF = 'feat/two-stage-llm-scoring'  # Use 'main' after the feature PR is merged.
REPO_DIR = Path('/content/interview_scoring_system')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Reusing existing checkout; REPO_REF is not applied to existing files.')
if not (REPO_DIR / 'notebook_support.py').is_file():
    raise RuntimeError('This checkout lacks notebook support. Select the feature branch, or main after merge, without discarding your work.')
print('Repository:', REPO_DIR)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)


## Configure the provider and data

Edit `llm_config.json` in the checkout, or upload a private configuration and set `CONFIG_PATH` below to it. Change `primary`, optional `secondary`, review thresholds/weights, input CSV, rules JSON, and output path. Relative paths are based on the configuration file's directory.

For configured cloud roles, add their keys to Colab Secrets under the profile's `api_key_env` name. Environment variables and `.key` files take priority. Only configured roles are considered; missing secondary Secrets produce a warning but do not block the primary. The wrapper prepares an explicitly configured secondary key for the child even if the primary later needs no review. Local roles do not request Secrets. If two roles use different key files, give them distinct `api_key_env` names.

A hosted Colab runtime's `127.0.0.1` is the Colab machine, not your home computer. For a home llama.cpp/Ollama server, use the local notebook or a local runtime. Do not publicly expose an unauthenticated inference server. Free-tier account settings still apply; see `HUN_CONFIGURATION.md`.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from notebook_support import load_notebook_config, run_notebook

CONFIG_PATH = REPO_DIR / 'llm_config.json'
config = load_notebook_config(CONFIG_PATH, use_colab_secrets=True)
print('Provider:', config.llm.provider, '| Model:', config.llm.model)
print('Output:', config.output_csv)
# Never print the secret value or store it in notebook cells.


## Score and inspect results

The shared helper runs `app.py --config ...` with the same two-stage policy, validation and shared request cap as the CLI. Both valid scores, their written evaluations, weighted result and normalized disagreement are saved. No common confidence is invented for an average. Secrets are passed to the child environment, not disk or arguments. `app.py` is never rewritten.

Budget exhaustion returns saved partial results with a warning. Other errors stop the cell without loading an old CSV. Every execution starts a new run with a new application request budget; provider quotas are separate.

In [ ]:
result_df = run_notebook(CONFIG_PATH, REPO_DIR, use_colab_secrets=True)
print('Partial results:', result_df.attrs['partial_results'])
result_df.head()


## Export and privacy

The complete CSV is already saved to the configured output path. Optionally download it using the commented cell below. Avoid printing the full log or credentials; displayed interview answers may still be sensitive. Clear notebook outputs before sharing or committing.

Compare a sample with human-scored answers before research use, particularly for low-confidence rows.

In [ ]:
# Optional: uncomment to download the CSV in Colab.
# from google.colab import files
# files.download(result_df.attrs['output_csv'])
